In [5]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from src.dataset import get_dataloaders
from src.config  import IMAGE_SIZE, PROCESSED_DIR

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else "cpu"
)
print(f"Gerät: {device}")

loaders = get_dataloaders(
    data_dir    = PROCESSED_DIR,
    img_size    = IMAGE_SIZE[0],
    batch_size  = 32,
    num_workers = 0,
    pin_memory  = False,
)
print("DataLoader bereit ✓")

Gerät: cpu
[WARN] Keine Bilder in: d:\Maturaarbeit\malaria-ai\notebooks\..\data\processed\train\healthy
[WARN] Keine Bilder in: d:\Maturaarbeit\malaria-ai\notebooks\..\data\processed\train\infected


RuntimeError: Keine Bilder in 'd:\Maturaarbeit\malaria-ai\notebooks\..\data\processed\train' gefunden.
Erlaubte Endungen: frozenset({'.jpeg', '.jpg', '.tiff', '.png', '.bmp', '.tif'})
Erwartete Unterordner: 'infected/' und 'healthy/'

In [4]:
import torchvision.models as models
from src.model import build_model

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_summary(model, name):
    dummy  = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[0])
    out    = model(dummy)
    n_param = count_parameters(model)
    print(f"  {name:<30s} | Parameter: {n_param:>10,} | Output: {list(out.shape)}")

print("=" * 65)
print("Modell-Übersicht")
print("=" * 65)

our_model   = build_model()
resnet18    = models.resnet18(weights="DEFAULT")
resnet18.fc = nn.Linear(512, 2)
efficientb0 = models.efficientnet_b0(weights="DEFAULT")
efficientb0.classifier[1] = nn.Linear(1280, 2)

model_summary(our_model,   "MalariaNet (build_model)")
model_summary(resnet18,    "ResNet-18 (Transfer)")
model_summary(efficientb0, "EfficientNet-B0 (Transfer)")
print("=" * 65)

ImportError: cannot import name 'MalariaNet' from 'src.model' (d:\Maturaarbeit\malaria-ai\notebooks\..\src\model.py)

In [ ]:
images, labels = next(iter(loaders["train"]))

our_model.eval()
with torch.no_grad():
    output = our_model(images)
    probs  = torch.softmax(output, dim=1)
    preds  = output.argmax(dim=1)

acc = (preds == labels).float().mean()
print(f"Batch-Accuracy (untrainiert): {acc:.2%}  ← sollte ~50% sein")
print(f"Output-Shape                : {list(output.shape)}")
print(f"Probs Beispiel              : {probs[0].detach().numpy()}")

In [ ]:
def quick_lr_test(model_fn, loader, lrs, n_batches=20):
    results   = {}
    criterion = nn.CrossEntropyLoss()

    for lr in lrs:
        m      = model_fn().to(device)
        opt    = torch.optim.Adam(m.parameters(), lr=lr)
        losses = []

        m.train()
        for i, (imgs, lbls) in enumerate(loader):
            if i >= n_batches:
                break
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad()
            loss = criterion(m(imgs), lbls)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        results[lr] = losses
        print(f"  LR {lr:.0e} → Endloss: {losses[-1]:.4f}")

    return results

print("Lernraten-Test (20 Batches pro LR)...")
lrs     = [1e-5, 1e-4, 1e-3, 1e-2]
results = quick_lr_test(build_model, loaders["train"], lrs)

plt.figure(figsize=(10, 5))
for lr, losses in results.items():
    plt.plot(losses, label=f"LR = {lr:.0e}", linewidth=2)

plt.title("Loss nach Lernrate", fontsize=13, fontweight="bold")
plt.xlabel("Batch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import cv2
from preprocessing.augmentation import plot_augmentation_examples

test_img_path = list(Path("../data/raw/infected").glob("*.png"))
if not test_img_path:
    test_img_path = list(Path("../data/raw/infected").glob("*.jpg"))

if test_img_path:
    img = cv2.imread(str(test_img_path[0]))
    if img is not None:
        plot_augmentation_examples(img, n_examples=6, extended=False)
        plot_augmentation_examples(img, n_examples=6, extended=True)
else:
    print("Keine Bilder in data/raw/infected gefunden.")

In [ ]:
def mini_train(model, loaders, epochs=3, lr=1e-3):
    model     = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history   = {"val_acc": []}

    for epoch in range(epochs):
        model.train()
        for imgs, lbls in loaders["train"]:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            criterion(model(imgs), lbls).backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in loaders["val"]:
                imgs, lbls = imgs.to(device), lbls.to(device)
                correct += (model(imgs).argmax(1) == lbls).sum().item()
                total   += lbls.size(0)

        acc = correct / total
        history["val_acc"].append(acc)
        print(f"  Epoche {epoch+1} | Val-Acc: {acc:.2%}")

    return history

print("build_model (ResNet50):")
hist_own = mini_train(build_model(), loaders, epochs=3)

print("
ResNet-18:")
rn18    = models.resnet18(weights="DEFAULT")
rn18.fc = nn.Linear(512, 2)
hist_rn = mini_train(rn18, loaders, epochs=3)

# Plot
epochs_range = range(1, 4)
plt.figure(figsize=(8, 5))
plt.plot(epochs_range, [a * 100 for a in hist_own["val_acc"]],
         "o-", color="#378ADD", linewidth=2, label="build_model (ResNet50)")
plt.plot(epochs_range, [a * 100 for a in hist_rn["val_acc"]],
         "s-", color="#E24B4A", linewidth=2, label="ResNet-18")
plt.title("Val-Accuracy Vergleich", fontsize=13, fontweight="bold")
plt.xlabel("Epoche")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()